In [1]:
# Phase 8 – RL Environment Design (GraphSAGE) | Cell 1: Setup + Load

from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pickle

PROJECT = Path(r"D:\univercity\omid_project\economics_project")

OUT_DIR  = PROJECT / "outputs" / "phase8"
OUT_DATA = OUT_DIR / "data"
OUT_TAB  = OUT_DIR / "tables"
OUT_FIG  = OUT_DIR / "figures"
OUT_LOG  = OUT_DIR / "logs"
for d in [OUT_DATA, OUT_TAB, OUT_FIG, OUT_LOG]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Phase 8 – RL Environment (GraphSAGE) | Cell 1")
print("=" * 60)
print(f"Timestamp    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Project Root : {PROJECT}")
print(f"Outputs      : {OUT_DIR}")
print("Main embedding: GraphSAGE (best Phase-7 benchmark)")

# Feature matrix with SR
fm = pd.read_pickle(PROJECT / "outputs" / "phase6" / "data" / "feature_matrix_with_sr_final.pkl")

# GraphSAGE embeddings (best by AUC/AP)
emb_gs = np.load(PROJECT / "outputs" / "phase7" / "data" / "emb_GraphSAGE.npy")
meta_emb = pd.read_csv(PROJECT / "outputs" / "phase7" / "data" / "embedding_meta.csv")

print(f"\nFeature Matrix : {fm.shape}")
print(f"GraphSAGE emb  : {emb_gs.shape}")
print(f"Emb meta       : {meta_emb.shape}")
print(f"Years          : {fm['year'].min()} – {fm['year'].max()}")
print(f"Countries      : {fm['iso2'].nunique()}")

print("\nCell 1 completed successfully.")

Phase 8 – RL Environment (GraphSAGE) | Cell 1
Timestamp    : 2026-09-29 19:15:43
Project Root : D:\univercity\omid_project\economics_project
Outputs      : D:\univercity\omid_project\economics_project\outputs\phase8
Main embedding: GraphSAGE (best Phase-7 benchmark)

Feature Matrix : (378, 41)
GraphSAGE emb  : (378, 32)
Emb meta       : (378, 2)
Years          : 2000 – 2020
Countries      : 18

Cell 1 completed successfully.


In [2]:
# Phase 8 – RL Environment (GraphSAGE) | Cell 2: Align State Arrays

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase8" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase8" / "tables"

print("=" * 60)
print("Phase 8 – RL Environment (GraphSAGE) | Cell 2: Align State")
print("=" * 60)

fm = pd.read_pickle(PROJECT / "outputs" / "phase6" / "data" / "feature_matrix_with_sr_final.pkl").copy()
meta = pd.read_csv(PROJECT / "outputs" / "phase7" / "data" / "embedding_meta.csv")
emb_gs = np.load(PROJECT / "outputs" / "phase7" / "data" / "emb_GraphSAGE.npy")

fm = fm.sort_values(["year", "iso2"]).reset_index(drop=True)
meta = meta.sort_values(["year", "iso2"]).reset_index(drop=True)

assert list(fm["year"]) == list(meta["year"]), "year mismatch"
assert list(fm["iso2"]) == list(meta["iso2"]), "iso2 mismatch"
print("Alignment FM ↔ GraphSAGE meta: PASSED")

nodes = sorted(fm["iso2"].unique().tolist())
years = sorted(fm["year"].unique().tolist())
print(f"Nodes ({len(nodes)}): {nodes}")
print(f"Years ({len(years)}): {years[0]}–{years[-1]}")

base_cols = [c for c in [
    "in_strength", "out_strength", "pagerank", "betweenness", "closeness",
    "rgdpmad", "debtgdp", "unemp", "iy",
    "credit_private_pct_gdp", "export_kusd", "trade_openness_kusd",
    "icio_output", "SR", "SR_network", "SR_fundamentals"
] if c in fm.columns]

X = fm[base_cols].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median(numeric_only=True))
mu, sd = X.mean(), X.std().replace(0, 1.0)
base_features = ((X - mu) / sd).values.astype(np.float32)

panel_index = fm[["year", "iso2", "iso3", "country_jst"]].copy()
sr_vec = fm["SR"].values.astype(np.float32)

np.save(OUT_DATA / "base_features.npy", base_features)
np.save(OUT_DATA / "emb_GraphSAGE.npy", emb_gs.astype(np.float32))
np.save(OUT_DATA / "sr_vector.npy", sr_vec)
panel_index.to_csv(OUT_DATA / "panel_index.csv", index=False)
pd.DataFrame({"iso2": nodes}).to_csv(OUT_DATA / "node_list.csv", index=False)
pd.DataFrame({"feature": base_cols}).to_csv(OUT_TAB / "base_feature_names.csv", index=False)

# optional other embeddings for sensitivity later
for name, path in {
    "TGN": PROJECT / "outputs" / "phase7" / "data" / "emb_TGN_stable.npy",
    "TGAT": PROJECT / "outputs" / "phase7" / "data" / "emb_TGAT.npy",
}.items():
    if path.exists():
        np.save(OUT_DATA / f"emb_{name}.npy", np.load(path).astype(np.float32))
        print(f"Also saved emb_{name}.npy (optional)")

print(f"\nbase_features : {base_features.shape}")
print(f"emb_GraphSAGE : {emb_gs.shape}")
print(f"sr_vector     : {sr_vec.shape}")
print(f"Main embedding file: emb_GraphSAGE.npy")
print("\nCell 2 completed successfully.")

Phase 8 – RL Environment (GraphSAGE) | Cell 2: Align State
Alignment FM ↔ GraphSAGE meta: PASSED
Nodes (18): ['AU', 'BE', 'CA', 'CH', 'DE', 'DK', 'ES', 'FI', 'FR', 'GB', 'IE', 'IT', 'JP', 'NL', 'NO', 'PT', 'SE', 'US']
Years (21): 2000–2020
Also saved emb_TGN.npy (optional)
Also saved emb_TGAT.npy (optional)

base_features : (378, 16)
emb_GraphSAGE : (378, 32)
sr_vector     : (378,)
Main embedding file: emb_GraphSAGE.npy

Cell 2 completed successfully.


In [3]:
# Phase 8 – RL Environment (GraphSAGE) | Cell 3: Env Class + module

from pathlib import Path
import textwrap
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase8" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase8" / "tables"
ENV_PATH = OUT_DATA / "economics_env.py"

print("=" * 60)
print("Phase 8 – RL Environment (GraphSAGE) | Cell 3: Env Class")
print("=" * 60)

env_source = textwrap.dedent('''
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class EconomicsNetworkEnv(gym.Env):
    """Phase-8 economics RL environment (3 levels). Embedding: GraphSAGE."""
    metadata = {"render_modes": []}
    LEVEL_CFG = {
        "normal":   {"obs_noise": 0.01, "action_scale": 0.05, "sr_weight": 1.0, "stability_weight": 0.3, "cost_weight": 0.05, "shock_prob": 0.05, "shock_scale": 0.02, "max_steps": 40},
        "moderate": {"obs_noise": 0.03, "action_scale": 0.04, "sr_weight": 1.2, "stability_weight": 0.5, "cost_weight": 0.10, "shock_prob": 0.10, "shock_scale": 0.04, "max_steps": 40},
        "strict":   {"obs_noise": 0.06, "action_scale": 0.03, "sr_weight": 1.5, "stability_weight": 0.8, "cost_weight": 0.20, "shock_prob": 0.15, "shock_scale": 0.07, "max_steps": 40},
    }

    def __init__(self, base_features, embeddings, sr_vector, panel_index, nodes,
                 level="normal", reward_scale=1.0, seed=None):
        super().__init__()
        assert level in self.LEVEL_CFG
        self.cfg = self.LEVEL_CFG[level]
        self.level = level
        self.reward_scale = reward_scale
        self.base_features = base_features.astype(np.float32)
        self.embeddings = embeddings.astype(np.float32)
        self.sr_vector = sr_vector.astype(np.float32)
        self.panel_index = panel_index.reset_index(drop=True)
        self.nodes = list(nodes)
        self.feat_dim = base_features.shape[1]
        self.emb_dim = embeddings.shape[1]
        self.max_steps = self.cfg["max_steps"]
        obs_dim = (self.feat_dim * 2) + self.emb_dim + 5
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(4,), dtype=np.float32)
        self.rng = np.random.default_rng(seed)
        self.current_features = None
        self.current_sr = None
        self.initial_sr = None
        self.step_count = 0
        self._year_indices = None

    def _sample_year_block(self):
        years = self.panel_index["year"].unique()
        y = int(self.rng.choice(years))
        idx = self.panel_index.index[self.panel_index["year"] == y].to_numpy()
        sub = self.panel_index.loc[idx].copy()
        sub["_ord"] = sub["iso2"].map({n: i for i, n in enumerate(self.nodes)})
        return sub.sort_values("_ord").index.to_numpy()

    def _get_obs(self):
        feat = self.current_features
        obs = np.concatenate([
            feat.mean(0), feat.std(0) + 1e-6,
            self.embeddings[self._year_indices].mean(0),
            np.array([
                float(np.mean(self.current_sr)),
                float(np.std(self.current_sr) + 1e-6),
                float(np.min(self.current_sr)),
                float(np.max(self.current_sr)),
            ], dtype=np.float32),
            np.array([self.step_count / max(self.max_steps, 1)], dtype=np.float32),
        ]).astype(np.float32)
        return obs + self.rng.normal(0.0, self.cfg["obs_noise"], size=obs.shape).astype(np.float32)

    def reset(self, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self._year_indices = self._sample_year_block()
        self.current_features = self.base_features[self._year_indices].copy()
        self.current_sr = self.sr_vector[self._year_indices].copy()
        self.initial_sr = self.current_sr.copy()
        self.step_count = 0
        return self._get_obs(), {}

    def step(self, action):
        action = np.clip(np.asarray(action, dtype=np.float32), -1.0, 1.0)
        s = self.cfg["action_scale"]
        tax, subsidy, invest, credit = action * s
        self.current_features = self.current_features + ((subsidy + invest + credit) - 0.5 * tax) * 0.1
        self.current_sr = np.clip(
            self.current_sr + (0.4 * invest + 0.3 * credit + 0.2 * subsidy - 0.3 * tax),
            0.0, 1.5,
        )
        if self.rng.random() < self.cfg["shock_prob"]:
            shock = self.rng.normal(0.0, self.cfg["shock_scale"], size=self.current_sr.shape)
            self.current_sr = np.clip(self.current_sr + shock, 0.0, 1.5)
            self.current_features = self.current_features + float(shock.mean()) * 0.05
        self.step_count += 1
        sr_now = float(np.mean(self.current_sr))
        sr_imp = sr_now - float(np.mean(self.initial_sr))
        reward = self.reward_scale * (
            self.cfg["sr_weight"] * sr_imp
            + self.cfg["stability_weight"] * (-float(np.std(self.current_sr)))
            + self.cfg["cost_weight"] * (-float(np.mean(np.abs(action))))
        )
        trunc = self.step_count >= self.max_steps
        info = {"sr_mean": sr_now, "sr_improve": sr_imp, "level": self.level, "step": self.step_count}
        return self._get_obs(), float(reward), False, trunc, info
''')

ENV_PATH.write_text(env_source, encoding="utf-8")
print(f"Saved env module → {ENV_PATH}")

import sys
if str(OUT_DATA) not in sys.path:
    sys.path.insert(0, str(OUT_DATA))
# reload if previously imported
import importlib
import economics_env
importlib.reload(economics_env)
from economics_env import EconomicsNetworkEnv

base = np.load(OUT_DATA / "base_features.npy")
emb = np.load(OUT_DATA / "emb_GraphSAGE.npy")  # main
sr = np.load(OUT_DATA / "sr_vector.npy")
panel = pd.read_csv(OUT_DATA / "panel_index.csv")
nodes = pd.read_csv(OUT_DATA / "node_list.csv")["iso2"].tolist()

for level in ["normal", "moderate", "strict"]:
    env = EconomicsNetworkEnv(base, emb, sr, panel, nodes, level=level, seed=0)
    obs, _ = env.reset(seed=0)
    total_r = 0.0
    for _ in range(5):
        a = env.action_space.sample()
        obs, r, term, trunc, info = env.step(a)
        total_r += r
    print(f"{level:10s} | obs={obs.shape} | emb_dim={env.emb_dim} | reward_5steps={total_r:.4f} | sr={info['sr_mean']:.4f}")

pd.DataFrame([{"level": k, **v} for k, v in EconomicsNetworkEnv.LEVEL_CFG.items()]).to_csv(
    OUT_TAB / "env_level_configs.csv", index=False
)
print("\nCell 3 completed successfully.")

Phase 8 – RL Environment (GraphSAGE) | Cell 3: Env Class
Saved env module → D:\univercity\omid_project\economics_project\outputs\phase8\data\economics_env.py
normal     | obs=(69,) | emb_dim=32 | reward_5steps=-0.1525 | sr=0.4204
moderate   | obs=(69,) | emb_dim=32 | reward_5steps=-0.4144 | sr=0.3829
strict     | obs=(69,) | emb_dim=32 | reward_5steps=-1.2435 | sr=0.3042

Cell 3 completed successfully.


In [4]:
# Phase 8 – RL Environment (GraphSAGE) | Cell 4: Random baseline + Finalize

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import pickle

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase8" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase8" / "tables"

if str(OUT_DATA) not in sys.path:
    sys.path.insert(0, str(OUT_DATA))
from economics_env import EconomicsNetworkEnv

print("=" * 60)
print("Phase 8 – RL Environment (GraphSAGE) | Cell 4: Baseline + Save")
print("=" * 60)

base = np.load(OUT_DATA / "base_features.npy")
emb = np.load(OUT_DATA / "emb_GraphSAGE.npy")  # main embedding
sr = np.load(OUT_DATA / "sr_vector.npy")
panel = pd.read_csv(OUT_DATA / "panel_index.csv")
nodes = pd.read_csv(OUT_DATA / "node_list.csv")["iso2"].tolist()

def eval_random(level, n_episodes=20, seed=0):
    env = EconomicsNetworkEnv(base, emb, sr, panel, nodes, level=level, seed=seed)
    rewards, sr_imps = [], []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        ep_r, last_info = 0.0, {}
        while True:
            a = env.action_space.sample()
            obs, r, term, trunc, info = env.step(a)
            ep_r += r
            last_info = info
            if term or trunc:
                break
        rewards.append(ep_r)
        sr_imps.append(last_info.get("sr_improve", np.nan))
    return {
        "level": level,
        "agent": "Random",
        "n_episodes": n_episodes,
        "mean_reward": float(np.mean(rewards)),
        "std_reward": float(np.std(rewards)),
        "mean_sr_improve": float(np.nanmean(sr_imps)),
        "std_sr_improve": float(np.nanstd(sr_imps)),
    }

rows = [eval_random(lv) for lv in ["normal", "moderate", "strict"]]
base_df = pd.DataFrame(rows)
print(base_df.to_string(index=False))
base_df.to_csv(OUT_TAB / "random_baseline_3levels.csv", index=False)

payload = {
    "base_features_path": str(OUT_DATA / "base_features.npy"),
    "emb_main_path": str(OUT_DATA / "emb_GraphSAGE.npy"),
    "emb_name": "GraphSAGE",
    "sr_vector_path": str(OUT_DATA / "sr_vector.npy"),
    "panel_index_path": str(OUT_DATA / "panel_index.csv"),
    "node_list_path": str(OUT_DATA / "node_list.csv"),
    "env_module_path": str(OUT_DATA / "economics_env.py"),
    "levels": list(EconomicsNetworkEnv.LEVEL_CFG.keys()),
    "action_names": ["tax", "subsidy", "investment", "credit"],
    "main_embedding": "GraphSAGE",
}
with open(OUT_DATA / "phase8_env_payload.pkl", "wb") as f:
    pickle.dump(payload, f)

print(f"\nMain embedding: GraphSAGE")
print(f"Saved payload → {OUT_DATA / 'phase8_env_payload.pkl'}")
print("\nCell 4 completed successfully.")
print("PHASE 8 (GraphSAGE) COMPLETED.")

Phase 8 – RL Environment (GraphSAGE) | Cell 4: Baseline + Save
   level  agent  n_episodes  mean_reward  std_reward  mean_sr_improve  std_sr_improve
  normal Random          20    -3.619308    2.688691        -0.012623        0.116473
moderate Random          20    -4.894765    2.697490         0.015127        0.088979
  strict Random          20   -10.245214    1.964775        -0.007482        0.063845

Main embedding: GraphSAGE
Saved payload → D:\univercity\omid_project\economics_project\outputs\phase8\data\phase8_env_payload.pkl

Cell 4 completed successfully.
PHASE 8 (GraphSAGE) COMPLETED.
